In [2]:
import pandas as pd
import csv
from pathlib import Path

# Paths will be added to config.py later
DATA_PATH = "data/raw/ACLF_2026-02-12_MDAT.csv"
OUTPUT_DIR = Path("data\processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DD = "metadata/processed/data_dictionary.csv"

<>:7: SyntaxWarning: invalid escape sequence '\p'
<>:7: SyntaxWarning: invalid escape sequence '\p'
C:\Users\saram\AppData\Local\Temp\ipykernel_20540\2081829637.py:7: SyntaxWarning: invalid escape sequence '\p'
  OUTPUT_DIR = Path("data\processed")


In [3]:
data_original = pd.read_csv(
    DATA_PATH,
    dtype="object",
    sep=";",
    skiprows=7
)

dd = pd.read_csv(
    DD,
    dtype="object",
    sep=";"
)

In [4]:
data_original.columns = data_original.columns.str.strip()
dd.columns = dd.columns.str.strip()

In [5]:
data_cleaned = data_original.rename(columns={
    "Episode number": "Episode",
    "Episode date": "Episode_Date",
    "Episode description": "Episode_Description",
})

In [6]:
data_cleaned["PID"] = data_cleaned["PID"].astype(str).str.strip()
data_cleaned["Episode"] = data_cleaned["Episode"].astype(str).str.strip()
data_cleaned["Episode_Date"] = data_cleaned["Episode_Date"].astype(str).str.strip()

dd["source_id"] = dd["source_id"].astype(str).str.strip()
dd["form_type"] = dd["form_type"].astype(str).str.strip().str.lower()

In [7]:
dd["form_type"].value_counts(dropna=False)

form_type
longitudinal    302
basic           158
Name: count, dtype: int64

In [8]:
basic_ids = dd.loc[
    dd["form_type"].eq("basic"),
    "source_id"
].dropna().unique().tolist()

longitudinal_ids = dd.loc[
    dd["form_type"].eq("longitudinal"),
    "source_id"
].dropna().unique().tolist()

In [9]:
wide_cols = set(data_cleaned.columns)

basic_ids_in_data = [x for x in basic_ids if x in wide_cols]
longitudinal_ids_in_data = [x for x in longitudinal_ids if x in wide_cols]

missing_basic_ids = [x for x in basic_ids if x not in wide_cols]
missing_longitudinal_ids = [x for x in longitudinal_ids if x not in wide_cols]

print("Basic in data:", len(basic_ids_in_data))
print("Longitudinal in data:", len(longitudinal_ids_in_data))
print("Missing basic:", len(missing_basic_ids))
print("Missing longitudinal:", len(missing_longitudinal_ids))

Basic in data: 156
Longitudinal in data: 295
Missing basic: 2
Missing longitudinal: 7


In [10]:
data_work = data_cleaned.copy()

# Treat empty strings as missing
data_work["Episode"] = data_work["Episode"].replace({"": pd.NA, "nan": pd.NA})
data_work["Episode_Date"] = data_work["Episode_Date"].replace({"": pd.NA, "nan": pd.NA})

# Forward-fill episode context within each patient
data_work[["Episode", "Episode_Date"]] = (
    data_work
    .groupby("PID")[["Episode", "Episode_Date"]]
    .ffill()
)

In [11]:
data_work["IX"] = (
    data_work
    .groupby(["PID", "Episode", "Episode_Date"])
    .cumcount()
    .add(1)
)

In [12]:
data_work[["PID", "Episode", "Episode_Date", "IX"]].head(30)

,PID,Episode,Episode_Date,IX
0,1,NaN,NaN,NaN
1,10,1,26/11/2020,1.0
2,100,1,27/09/2021,1.0
3,100,1,27/09/2021,2.0
4,100,2,20/10/2021,1.0
5,100,2,20/10/2021,2.0
6,101,1,27/09/2021,1.0
7,101,1,27/09/2021,2.0
8,101,2,31/01/2023,1.0
9,101,2,31/01/2023,2.0


In [13]:
data_work = data_work[
    ~data_work["Location"]
    .astype(str)
    .str.strip()
    .str.lower()
    .str.contains("test location", na=False)
]

In [14]:
non_data_cols = ["PID", "Location", "Episode", "Episode_Date", "Episode_Description", "IX"]
data_cols = [c for c in data_work.columns if c not in non_data_cols]
data_work[data_cols] = data_work[data_cols].replace({"": pd.NA, " ": pd.NA})
data_work = data_work[
    data_work[data_cols].notna().any(axis=1)
]
valid_pids = (
    data_work
    .groupby("PID")[data_cols]
    .apply(lambda df: df.notna().any().any())
)

valid_pids = valid_pids[valid_pids].index

data_work = data_work[data_work["PID"].isin(valid_pids)]

C:\Users\saram\AppData\Local\Temp\ipykernel_20540\2159394179.py:3: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data_work[data_cols] = data_work[data_cols].replace({"": pd.NA, " ": pd.NA})


In [17]:
data_work.shape


(1212, 507)

In [16]:
data_work

,PID,patient repeat number,Episode,Location,Episode_Date,Episode_Description,Unique ID (Form_Record_DataElement):,X_153_0_2,X_222_0_42,X_222_0_43,...,X_223_6_18,X_236_0_338,X_236_0_339,X_236_0_340,X_236_0_341,X_236_0_342,X_236_0_343,X_236_0_344,Unnamed: 505,IX
1,10,2,1,Goethe University of Frankfurt,26/11/2020,d0,NaN,NaN,Frankfurt University Hospital,0003,...,NaN,Alive,NaN,NaN,NaN,NaN,No,NaN,NaN,1.0
2,100,3,1,Goethe University of Frankfurt,27/09/2021,d0,NaN,NaN,GUF,0085,...,NaN,Alive,NaN,NaN,NaN,NaN,No,NaN,NaN,1.0
3,100,3,1,Goethe University of Frankfurt,27/09/2021,d0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0
4,100,3,2,Goethe University of Frankfurt,20/10/2021,Re-admission1,NaN,NaN,NaN,NaN,...,NaN,Alive,NaN,NaN,NaN,NaN,Yes,01.11.2022,NaN,1.0
5,100,3,2,Goethe University of Frankfurt,20/10/2021,Re-admission1,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1260,332,309,0,Goethe University of Frankfurt,NaN,NaN,No data,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1261,333,310,0,Goethe University of Frankfurt,NaN,NaN,No data,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1262,334,311,0,Goethe University of Frankfurt,NaN,NaN,No data,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1263,335,312,0,Goethe University of Frankfurt,NaN,NaN,No data,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [18]:
# Empty strings -> NA
data_work["Episode"] = data_work["Episode"].replace({"": pd.NA, "nan": pd.NA})
data_work["Episode_Date"] = data_work["Episode_Date"].replace({"": pd.NA, "nan": pd.NA})

# Forward-fill episode context within patient
data_work[["Episode", "Episode_Date"]] = (
    data_work
    .groupby("PID")[["Episode", "Episode_Date"]]
    .ffill()
)

# Create repeat index within patient + episode + episode date
data_work["IX"] = (
    data_work
    .groupby(["PID", "Episode", "Episode_Date"])
    .cumcount()
    .add(1)
)

In [19]:
data_work[["PID", "Episode", "Episode_Date", "IX"]].head(30)

,PID,Episode,Episode_Date,IX
1,10,1,26/11/2020,1.0
2,100,1,27/09/2021,1.0
3,100,1,27/09/2021,2.0
4,100,2,20/10/2021,1.0
5,100,2,20/10/2021,2.0
6,101,1,27/09/2021,1.0
7,101,1,27/09/2021,2.0
8,101,2,31/01/2023,1.0
9,101,2,31/01/2023,2.0
10,102,1,29/09/2021,1.0


In [20]:
# Drop "No Data" patients

uid_col = "Unique ID (Form_Record_DataElement):"

data_work = data_work[
    ~data_work[uid_col]
    .astype(str)
    .str.strip()
    .str.lower()
    .eq("no data")
]

In [21]:
data_work

,PID,patient repeat number,Episode,Location,Episode_Date,Episode_Description,Unique ID (Form_Record_DataElement):,X_153_0_2,X_222_0_42,X_222_0_43,...,X_223_6_18,X_236_0_338,X_236_0_339,X_236_0_340,X_236_0_341,X_236_0_342,X_236_0_343,X_236_0_344,Unnamed: 505,IX
1,10,2,1,Goethe University of Frankfurt,26/11/2020,d0,NaN,NaN,Frankfurt University Hospital,0003,...,NaN,Alive,NaN,NaN,NaN,NaN,No,NaN,NaN,1.0
2,100,3,1,Goethe University of Frankfurt,27/09/2021,d0,NaN,NaN,GUF,0085,...,NaN,Alive,NaN,NaN,NaN,NaN,No,NaN,NaN,1.0
3,100,3,1,Goethe University of Frankfurt,27/09/2021,d0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0
4,100,3,2,Goethe University of Frankfurt,20/10/2021,Re-admission1,NaN,NaN,NaN,NaN,...,NaN,Alive,NaN,NaN,NaN,NaN,Yes,01.11.2022,NaN,1.0
5,100,3,2,Goethe University of Frankfurt,20/10/2021,Re-admission1,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1223,98,275,2,Goethe University of Frankfurt,26/01/2022,Re-admission1,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0
1224,98,275,2,Goethe University of Frankfurt,26/01/2022,Re-admission1,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.0
1225,99,276,1,Goethe University of Frankfurt,24/09/2021,d0,NaN,NaN,GUF,0082,...,NaN,Alive,NaN,NaN,NaN,NaN,No,NaN,NaN,1.0
1226,99,276,1,Goethe University of Frankfurt,24/09/2021,d0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0


In [22]:
# checking if rows without episode data contain longitudinal values

missing_episode_rows = data_work[
    data_work["Episode"].isna() | data_work["Episode_Date"].isna()
]

missing_episode_rows[[
    "PID",
    "Location",
    "Episode",
    "Episode_Date",
    "Episode_Description",
    "Unique ID (Form_Record_DataElement):"
]].head(20)

missing_longitudinal_values = missing_episode_rows[
    missing_episode_rows[longitudinal_ids_in_data].notna().any(axis=1)
]

missing_longitudinal_values.shape

(0, 507)

In [23]:
missing_basic_values = missing_episode_rows[
    missing_episode_rows[basic_ids_in_data].notna().any(axis=1)
]

missing_basic_values.shape

(16, 507)

### Melting Basic x Londigutinal

# Basic rows can exist without episode information. Longitudinal rows must have episode information.

In [24]:
basic_source = data_work.copy()

longitudinal_source = data_work.dropna(
    subset=["Episode", "Episode_Date"]
).copy()

In [25]:
# Melt basic variables - for those the only true identifier is PID, we dont use Episode or Episode_Date or IX

basic_long = pd.melt(
    basic_source,
    id_vars=["PID"],
    value_vars=basic_ids_in_data,
    var_name="source_id",
    value_name="source_value"
)

In [26]:
# Drop empty duplicate rows when the same PID/source_id combination already has a non-missing value
# Drops empty rows preserving true missingess

basic_long["source_value"] = basic_long["source_value"].replace(
    {"": pd.NA, " ": pd.NA}
)

basic_long["has_any_value"] = (
    basic_long
    .groupby(["PID", "source_id"])["source_value"]
    .transform(lambda x: x.notna().any())
)


basic_long_clean = basic_long[
    basic_long["source_value"].notna() |
    ~basic_long["has_any_value"]
].copy()

In [28]:
basic_long_clean = basic_long_clean.drop_duplicates(
    subset=["PID", "source_id", "source_value"]
)

basic_long_clean["Episode"] = "basic"
basic_long_clean["Episode_Date"] = pd.NA
basic_long_clean["IX"] = 1

In [30]:
# Melting Longitudinal
longitudinal_long = pd.melt(
    longitudinal_source,
    id_vars=["PID", "Episode", "Episode_Date"],
    value_vars=longitudinal_ids_in_data,
    var_name="source_id",
    value_name="source_value"
)


long_key = ["PID", "Episode", "Episode_Date", "source_id"]
longitudinal_long["source_value"] = longitudinal_long["source_value"].replace(
    {"": pd.NA, " ": pd.NA}
)

longitudinal_long["has_any_value"] = (
    longitudinal_long
    .groupby(long_key)["source_value"]
    .transform(lambda x: x.notna().any())
)

longitudinal_long_clean = longitudinal_long[
    longitudinal_long["source_value"].notna() |
    ~longitudinal_long["has_any_value"]
].copy()


# Assign IX
longitudinal_long_clean["IX"] = (
    longitudinal_long_clean
    .groupby(long_key)
    .cumcount()
    .add(1)
)

In [31]:
basic_long_clean = basic_long_clean[[
    "PID", "Episode", "Episode_Date", "IX", "source_id", "source_value"
]]

longitudinal_long_clean = longitudinal_long_clean[[
    "PID", "Episode", "Episode_Date", "IX", "source_id", "source_value"
]]

export_long_clean = pd.concat(
    [basic_long_clean, longitudinal_long_clean],
    ignore_index=True
)

In [35]:
print(export_long_clean.shape)
print(export_long_clean["PID"].nunique())
print(export_long_clean["source_id"].nunique())

export_long_clean.to_csv("data/processed/export_long_clean.csv", sep=";")

(247704, 6)
266
451


In [33]:
EXPORT_LONG_PATH = "data/processed/Export_long.csv"
export_long=pd.read_csv(EXPORT_LONG_PATH, sep=";", dtype="object")

old_ids = set(export_long["source_id"].astype(str).str.strip())
new_ids = set(export_long_clean["source_id"].astype(str).str.strip())

missing_in_new = sorted(old_ids - new_ids)
new_only = sorted(new_ids - old_ids)

print("Old source_ids:", len(old_ids))
print("New source_ids:", len(new_ids))
print("Missing in new:", len(missing_in_new))
print("New only:", len(new_only))

missing_in_new_dd = dd[
    dd["source_id"].astype(str).str.strip().isin(missing_in_new)
][[
    "source_id",
    "form_type",
    "Form",
    "dataelement_designation"
]]

missing_in_new_dd

Old source_ids: 498
New source_ids: 451
Missing in new: 47
New only: 0


,source_id,form_type,Form,dataelement_designation
